# Drone or Bird? — training on Colab
Author: Mariah Habak

Runtime → Change runtime type → **T4 GPU**. Run the cells in order.
The dataset zip from Mendeley must be in Google Drive at `MyDrive/skytrace_data/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/mariahabak/skytrace.git
%cd /content/skytrace
!pip freeze > requirements-colab.txt
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Unzip the dataset to Colab's fast local disk (also unzips zips inside the zip).
import glob, os, zipfile
target = 'data/raw/bird_vs_drone'
os.makedirs(target, exist_ok=True)
zips = glob.glob('/content/drive/MyDrive/skytrace_data/*.zip')
print(zips)
while zips:
    z = zips.pop()
    with zipfile.ZipFile(z) as f:
        f.extractall(target)
    if z.startswith(target):
        os.remove(z)
    zips += glob.glob(target + '/**/*.zip', recursive=True)
print(sum(len(f) for _, _, f in os.walk(target)), 'files')

In [ ]:
# STEP 1 — look at the data. Check the class ids per folder before going on.
!python -m dronebird.audit data/raw/bird_vs_drone

In [ ]:
!python -m pytest -q
!python -m dronebird.prepare configs/data.json

In [ ]:
for name in ['majority', 'small_cnn', 'resnet18', 'resnet18_distance']:
    !python -m dronebird.run configs/{name}.json --stage fit
!python -m dronebird.tables --split val

In [ ]:
# Save results (with the seed-0 checkpoints for the demo) to Drive and download them.
!zip -qr results.zip results data/processed/split.json data/processed/summary.json data/processed/figures requirements-colab.txt audit.json results/*/seed0/best.pt
!cp results.zip /content/drive/MyDrive/skytrace_data/
from google.colab import files; files.download('results.zip')